# 01 — Exploratory Data Analysis

Characterizes the traffic signal before any modeling: the spatial
distribution of total traffic across all 10,000 squares, the time-domain
behaviour of the 5 target squares, and the autocorrelation/stationarity
structure of the highest-traffic square. These findings directly motivate
the model choices in `02_sarima.ipynb` and `03_lstm_tcn.ipynb` (seasonal
period, lag horizon, sequence length).

Requires `00_data_pipeline.ipynb` to have been run first (reads
`data/processed/square_totals.csv`, `target_squares_timeseries.csv`,
`target_squares.yaml`).

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import common
from common import OBSERVATION_START, SQUARE_TOTALS_PATH

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

meta = common.load_target_squares_meta()
totals = pd.read_csv(SQUARE_TOTALS_PATH, index_col="square_id")["internet_total"]
print("Top-3 squares by total internet traffic:", meta["top3_totals"])
print("Fixed squares (assignment brief):", meta["fixed_squares"])

## 1. Distribution of total traffic across all 10,000 squares

If traffic is roughly uniform across the city, a single "representative"
square would generalize well; if it's concentrated in a few hubs, the
model comparison should focus there (which is what the assignment's
"top-3 by total traffic" selection does).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(totals.values, bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of total Internet traffic per square")
axes[0].set_xlabel("Total internet traffic (observation period)")
axes[0].set_ylabel("Number of squares")

axes[1].boxplot(totals.values, vert=True)
axes[1].set_title("Boxplot of total Internet traffic per square")
axes[1].set_ylabel("Total internet traffic")
fig.tight_layout()
fig.savefig("figures/eda_traffic_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

dist_stats = {
    "n_squares": int(totals.shape[0]),
    "mean": float(totals.mean()),
    "median": float(totals.median()),
    "std": float(totals.std()),
    "min": float(totals.min()),
    "max": float(totals.max()),
    "skewness": float(totals.skew()),
    "kurtosis": float(totals.kurtosis()),
    "p90": float(totals.quantile(0.90)),
    "p99": float(totals.quantile(0.99)),
    "share_top_1pct_of_total": float(totals.sort_values(ascending=False).head(int(len(totals) * 0.01)).sum() / totals.sum()),
}
pd.DataFrame([dist_stats]).to_csv("results/data_summary.csv", index=False)
dist_stats

**Interpretation.** Total Internet traffic per square is **heavily
right-skewed** (skewness ≈ 4.24, excess kurtosis ≈ 24.3; mean ≈ 298,269 vs.
median ≈ 147,153 — the mean is roughly double the median). The top 1% of
squares (100 of 10,000) account for **~11.3%** of total citywide Internet
traffic. This is consistent with the well-known spatial heterogeneity of
urban mobile demand: a small number of business/transit hubs generate
disproportionate traffic relative to the long tail of residential/peripheral
squares. **Implication:** it's reasonable to focus modeling effort on the
highest-traffic squares (top-3), since they carry a disproportionate share
of the network load an operator would actually need to forecast accurately
for capacity planning.

## 2. First two weeks: top-3 squares + the two brief-required squares

Comparing the 3 highest-traffic squares against the 2 squares the assignment
brief specifically requires (4159, 4556) shows whether "high traffic" also
means "differently shaped" traffic, or just "the same daily pattern, scaled
up".

In [ ]:
five_squares = sorted(set(meta["top3_squares"]) | set(meta["fixed_squares"]))
two_weeks_end = pd.Timestamp(OBSERVATION_START) + pd.Timedelta(days=14)
print("Plotting first-two-weeks series for squares:", five_squares)

fig, axes = plt.subplots(len(five_squares), 1, figsize=(11, 2.3 * len(five_squares)), sharex=True)
for ax, sq in zip(axes, five_squares):
    s = common.load_square_series(sq)
    s = s[s.index < two_weeks_end]
    ax.plot(s.index, s.values, linewidth=0.8, color="darkorange")
    ax.set_ylabel(f"Sq {sq}", rotation=0, ha="right", va="center", fontsize=9)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Time")
fig.suptitle("Internet traffic, first two weeks of the observation period", y=1.01)
fig.tight_layout()
fig.savefig("figures/eda_five_squares_first_two_weeks.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretation.** All five squares show a clear **daily commuting
rhythm** — a trough around 04:00–06:00 and a broad daytime plateau/peak —
but differ substantially in *scale*, *peak shape*, and *weekday/weekend
contrast*:
- **Square 4159** shows the strongest weekday/weekend contrast of the five:
  visibly lower and flatter peaks on the weekend (Nov 9–10), consistent with
  a business/office-dominated area where demand is commute-driven.
- **Square 4556** has a comparatively "noisier", less sharply peaked profile
  with a higher baseline overnight, suggesting a more residential or
  mixed-use area with less pronounced day/night contrast.
- **Squares 5059, 5161, and 5259** (the top-3) show the largest peaks and
  are visibly correlated with each other in timing, consistent with them
  being geographically adjacent, high-density squares. **Square 5161** in
  particular shows two large, short-lived spikes (Nov 3 and Nov 10, both
  Sundays) that are 2–3× its typical Sunday peak — an anomaly worth flagging
  for the forecasting task, since a model trained mostly on "typical" days
  will systematically under-predict such events (revisited as a concrete
  failure case in `04_model_comparison.ipynb`).

## 3. Autocorrelation, seasonality, and stationarity of the top-traffic square

These three analyses directly determine modeling choices: the ACF/PACF
inform SARIMA's `(p, d, q)` order and the LSTM/TCN sequence length; the STL
decomposition confirms whether seasonality needs explicit modeling (Fourier
terms) or can be left implicit; the ADF test checks whether differencing is
required.

In [ ]:
top_square = meta["top3_squares"][0]
s = common.load_square_series(top_square)
print(f"Running ACF/PACF + STL decomposition + ADF test on top square {top_square}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(s, lags=288, ax=axes[0])   # 2 days of 10-min lags
axes[0].set_title(f"ACF - square {top_square}")
plot_pacf(s, lags=288, ax=axes[1], method="ywm")
axes[1].set_title(f"PACF - square {top_square}")
fig.tight_layout()
fig.savefig("figures/eda_acf_pacf_top1.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretation (ACF/PACF).** The ACF shows a strong, slowly-decaying
periodic pattern with peaks at lag ≈144 and ≈288 (i.e. 1 and 2 days at
10-minute resolution) and troughs at half-day offsets — a textbook signature
of dominant daily seasonality with no evidence of a shorter cycle. The PACF
cuts off sharply after 1–2 lags, indicating that, once daily seasonality is
accounted for, the remaining short-term dependence is low-order.
**Implication for modelling:** this justifies (i) using a *low-order*
autoregressive component for SARIMA combined with an explicit seasonal term
rather than a large `p`, and (ii) a sequence length of at least 144 steps
(1 day) for the LSTM/TCN so they can see a full seasonal cycle.

In [ ]:
stl = STL(s, period=144, robust=True).fit()
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
axes[0].plot(s.index, s.values, linewidth=0.7); axes[0].set_ylabel("Observed")
axes[1].plot(s.index, stl.trend, linewidth=0.9, color="green"); axes[1].set_ylabel("Trend")
axes[2].plot(s.index, stl.seasonal, linewidth=0.7, color="purple"); axes[2].set_ylabel("Seasonal (daily)")
axes[3].plot(s.index, stl.resid, linewidth=0.5, color="gray"); axes[3].set_ylabel("Residual")
fig.suptitle(f"STL decomposition (period=144, i.e. 1 day) - square {top_square}", y=1.01)
fig.tight_layout()
fig.savefig("figures/eda_stl_decomposition_top1.png", dpi=150, bbox_inches="tight")
plt.show()

adf_level = adfuller(s.dropna(), autolag="AIC")
adf_diff = adfuller(s.diff().dropna(), autolag="AIC")
stationarity_df = pd.DataFrame([
    {"series": "level", "adf_stat": adf_level[0], "p_value": adf_level[1],
     "n_lags_used": adf_level[2], "critical_5pct": adf_level[4]["5%"],
     "stationary_at_5pct": adf_level[1] < 0.05},
    {"series": "first_difference", "adf_stat": adf_diff[0], "p_value": adf_diff[1],
     "n_lags_used": adf_diff[2], "critical_5pct": adf_diff[4]["5%"],
     "stationary_at_5pct": adf_diff[1] < 0.05},
])
stationarity_df.to_csv("results/stationarity_tests.csv", index=False)
stationarity_df

**Interpretation (STL + ADF).** STL decomposition with a 144-step seasonal
period confirms a large, stable daily seasonal component and a
slowly-varying trend, with residuals that are small most of the time but
show sharp spikes aligned with the anomalous days noted above. The Augmented
Dickey-Fuller test on the raw level series rejects the unit-root null
(ADF statistic ≈ −13.75, p ≈ 1.1×10⁻²⁵; also rejected on the first
difference, statistic ≈ −9.79, p ≈ 6.5×10⁻¹⁷). **This is a useful but
partial signal**: ADF tests for a stochastic trend, not for seasonality, so
a "stationary" verdict here does not mean the series lacks structure to
model — it means differencing is *not* strictly required to remove a trend,
but the strong deterministic-like daily cycle still needs to be modeled
explicitly (via seasonal/Fourier terms for SARIMA, or a long-enough
sequence/receptive field for LSTM/TCN), which directly motivates the
modelling choices in the next two notebooks.

**Summary of what this EDA implies for modeling:**
| Finding | Modeling consequence |
|---|---|
| Heavy right-skew, top-1% dominate traffic | Focus on top-3 squares (justifies the assignment's selection) |
| Strong daily (144-step) periodicity, low-order residual dependence | SARIMA: low `(p,d,q)` + Fourier seasonal terms, not a full seasonal-ARIMA term |
| ACF/PACF cut off after 1-2 lags once seasonality removed | LSTM/TCN: sequence length ≥ 144 steps to see a full cycle |
| Stationary in level and first difference | `d=1` differencing is a safe, non-aggressive default for SARIMA |
| Occasional sharp spikes (e.g. square 5161, Sunday nights) | Expect this to be a concrete failure case in the model comparison |